# Eje 1 — Pandas básico: análisis de fincas productoras de café

**Dataset:** `fincas_produccion_cafe.csv` — 1012 observaciones y 8 variables (12 de ellas repetidas).

El recorrido cubre: carga del dataset, selección unidimensional y proyección multivariable, acceso por etiqueta con `.loc[]`, eliminación de duplicados, los distintos tipos de filtrado (booleano, `.isin()`, `.between()`, `.query()` y compuesto con métodos de texto), imputación de valores faltantes, creación de columnas con `.apply()`, renombrado y reordenamiento de atributos, y agrupación con métricas de producción (incluye una función propia `rango` y agrupación multinivel).


In [156]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


## 1. Carga del dataset

`index_col` **no** se usa a propósito: el índice queda como `RangeIndex` (0..n-1), que es lo que permite consultar por posición con `.loc[280]` más adelante, y `ID_Finca` permanece como columna para poder reordenarla en la sección 12.


In [157]:
df_fincas = pd.read_csv("fincas_produccion_cafe.csv", encoding="utf-8")

print("Dimensiones        :", df_fincas.shape)
print("Tipo de índice     :", type(df_fincas.index).__name__)
print("Filas repetidas    :", int(df_fincas.duplicated().sum()))
print("\nTipos de dato por columna:")
df_fincas.dtypes


Dimensiones        : (1012, 8)
Tipo de índice     : RangeIndex
Filas repetidas    : 12

Tipos de dato por columna:


ID_Finca                  str
Municipio                 str
Variedad                  str
Metodo_Beneficio          str
Area_Hectareas        float64
Kilos_Recolectados    float64
Altitud_MSNM            int64
Certificacion             str
dtype: object

## 2. Selección unidimensional y proyección multivariable

Una sola columna devuelve una **Serie** de pandas; varias columnas devuelven una **vista de tabla** (`DataFrame`).


In [158]:
# Una sola columna -> Serie de pandas
serie_variedades = df_fincas["Variedad"]

serie_variedades


0       Cenicafé 1
1       Cenicafé 1
2       Cenicafé 1
3       Cenicafé 1
4         Castillo
           ...    
1007    Cenicafé 1
1008    Cenicafé 1
1009    Cenicafé 1
1010      Colombia
1011      Castillo
Name: Variedad, Length: 1012, dtype: str

In [159]:
# Varias columnas -> vista de tabla (DataFrame)
vista_produccion = df_fincas[["Municipio", "Area_Hectareas", "Kilos_Recolectados"]]
vista_produccion


,Municipio,Area_Hectareas,Kilos_Recolectados
0,Chinchiná,3.0,4800.0
1,Manizales,5.5,2500.0
2,Manizales,1.5,1200.0
3,Risaralda,3.0,4800.0
4,Chinchiná,8.0,16000.0
...,...,...,...
1007,Manizales,5.5,2500.0
1008,Manizales,12.0,7500.0
1009,Palestina,5.5,4800.0
1010,Risaralda,12.0,NaN


## 3. Acceso por etiqueta con `.loc[]`

Consulta de todos los datos del predio situado en la **posición de índice 280**.


In [160]:
predio_280 = df_fincas.loc[280]
print("Tipo del resultado:", type(predio_280).__name__)
predio_280


Tipo del resultado: Series


ID_Finca               FIN-8281
Municipio             Risaralda
Variedad               Colombia
Metodo_Beneficio        Natural
Area_Hectareas              5.5
Kilos_Recolectados      16000.0
Altitud_MSNM               1821
Certificacion          Orgánico
Name: 280, dtype: object

## 4. Eliminación de filas duplicadas


In [161]:
filas_antes = len(df_fincas)
duplicados = int(df_fincas.duplicated().sum())

df_fincas = df_fincas.drop_duplicates().reset_index(drop=True)

print(f"Filas antes de limpiar            : {filas_antes}")
print(f"Observaciones repetidas eliminadas: {duplicados}")
print(f"Filas después de limpiar          : {len(df_fincas)}")
print("Duplicados restantes              :", int(df_fincas.duplicated().sum()))

Filas antes de limpiar            : 1012
Observaciones repetidas eliminadas: 12
Filas después de limpiar          : 1000
Duplicados restantes              : 0


## 5. Filtrado condicional básico — lotes a gran altura (`Altitud_MSNM > 1750`)


In [162]:
filtro_altura = df_fincas[df_fincas["Altitud_MSNM"] > 1750]
print("Lotes a gran altura (> 1750 msnm):", len(filtro_altura))
filtro_altura


Lotes a gran altura (> 1750 msnm): 355


,ID_Finca,Municipio,Variedad,Metodo_Beneficio,Area_Hectareas,Kilos_Recolectados,Altitud_MSNM,Certificacion
2,FIN-8003,Manizales,Cenicafé 1,Lavado Tradicional,1.5,1200.0,1959,Rainforest Alliance
3,FIN-8004,Risaralda,Cenicafé 1,Lavado Tradicional,3.0,4800.0,1767,Ninguna
4,FIN-8005,Chinchiná,Castillo,Natural,8.0,16000.0,1929,Fairtrade
12,FIN-8013,Risaralda,Borbon Rosado,Natural,5.5,4800.0,1993,Rainforest Alliance
13,FIN-8014,Manizales,Borbon Rosado,Natural,1.5,2500.0,2011,Fairtrade
...,...,...,...,...,...,...,...,...
980,FIN-8981,Palestina,Castillo,Natural,12.0,7500.0,1968,Fairtrade
981,FIN-8982,Risaralda,Colombia,Honey,8.0,4800.0,2020,Fairtrade
984,FIN-8985,Chinchiná,Geisha,Lavado Tradicional,12.0,1200.0,1780,Fairtrade
985,FIN-8986,Anserma,Borbon Rosado,Lavado Tradicional,8.0,4800.0,1874,Fairtrade


## 6. Filtrado categórico por lista (`.isin()`) — Anserma y Risaralda


In [163]:
filtro_municipios = df_fincas[df_fincas["Municipio"].isin(["Anserma", "Risaralda"])]
print("Lotes en Anserma o Risaralda:", len(filtro_municipios))
filtro_municipios["Municipio"].value_counts()


Lotes en Anserma o Risaralda: 387


Municipio
Anserma      194
Risaralda    193
Name: count, dtype: int64

## 7. Filtrado por intervalo numérico (`.between()`) — `Area_Hectareas` entre 3.0 y 8.0 ha


In [164]:
df_fincas[df_fincas["Area_Hectareas"].between(3.0, 8.0, inclusive="both")]


,ID_Finca,Municipio,Variedad,Metodo_Beneficio,Area_Hectareas,Kilos_Recolectados,Altitud_MSNM,Certificacion
0,FIN-8001,Chinchiná,Cenicafé 1,Honey,3.0,4800.0,1500,Ninguna
1,FIN-8002,Manizales,Cenicafé 1,Natural,5.5,2500.0,1284,Rainforest Alliance
3,FIN-8004,Risaralda,Cenicafé 1,Lavado Tradicional,3.0,4800.0,1767,Ninguna
4,FIN-8005,Chinchiná,Castillo,Natural,8.0,16000.0,1929,Fairtrade
6,FIN-8007,Anserma,Colombia,Honey,5.5,4800.0,1674,Rainforest Alliance
...,...,...,...,...,...,...,...,...
992,FIN-8993,Chinchiná,Castillo,Natural,3.0,2500.0,1508,Ninguna
994,FIN-8995,Chinchiná,Cenicafé 1,Natural,5.5,1200.0,1297,Fairtrade
995,FIN-8996,Anserma,Geisha,Natural,3.0,7500.0,1774,Rainforest Alliance
996,FIN-8997,Palestina,Borbon Rosado,Natural,3.0,4800.0,1263,Rainforest Alliance


## 8. Consultas directas con `.query()`


In [165]:
df_fincas.query("Certificacion == 'Orgánico' and Kilos_Recolectados > 5000")

,ID_Finca,Municipio,Variedad,Metodo_Beneficio,Area_Hectareas,Kilos_Recolectados,Altitud_MSNM,Certificacion
5,FIN-8006,Risaralda,Cenicafé 1,Honey,12.0,7500.0,1652,Orgánico
30,FIN-8031,Palestina,Castillo,Natural,18.5,16000.0,1838,Orgánico
34,FIN-8035,Chinchiná,Borbon Rosado,Natural,3.0,7500.0,1578,Orgánico
61,FIN-8062,Palestina,Geisha,Lavado Tradicional,1.5,11000.0,1885,Orgánico
69,FIN-8070,Palestina,Castillo,Lavado Tradicional,8.0,16000.0,1922,Orgánico
111,FIN-8112,Risaralda,Castillo,Honey,NaN,7500.0,1641,Orgánico
173,FIN-8174,Palestina,Colombia,Lavado Tradicional,NaN,7500.0,2018,Orgánico
187,FIN-8188,Manizales,Geisha,Lavado Tradicional,5.5,7500.0,1328,Orgánico
200,FIN-8201,Risaralda,Colombia,Honey,3.0,7500.0,1563,Orgánico
209,FIN-8210,Palestina,Geisha,Natural,5.5,7500.0,1540,Orgánico


## 9. Filtrado compuesto con métodos de texto

Condición lógica con el operador `&`: el beneficio debe **iniciar** con la palabra "Lavado" (`.str.startswith()`) **y** la variedad debe ser "Geisha".


In [166]:
cond = df_fincas["Metodo_Beneficio"].str.startswith("Lavado", na=False) & (df_fincas["Variedad"] == "Geisha")
df_fincas[cond]


,ID_Finca,Municipio,Variedad,Metodo_Beneficio,Area_Hectareas,Kilos_Recolectados,Altitud_MSNM,Certificacion
16,FIN-8017,Palestina,Geisha,Lavado Tradicional,3.0,4800.0,1837,Rainforest Alliance
18,FIN-8019,Manizales,Geisha,Lavado Tradicional,12.0,7500.0,1500,Fairtrade
38,FIN-8039,Anserma,Geisha,Lavado Tradicional,18.5,1200.0,1675,Rainforest Alliance
50,FIN-8051,Chinchiná,Geisha,Lavado Tradicional,3.0,7500.0,1689,Rainforest Alliance
61,FIN-8062,Palestina,Geisha,Lavado Tradicional,1.5,11000.0,1885,Orgánico
...,...,...,...,...,...,...,...,...
943,FIN-8944,Manizales,Geisha,Lavado Tradicional,5.5,NaN,1313,Rainforest Alliance
946,FIN-8947,Chinchiná,Geisha,Lavado Tradicional,NaN,11000.0,1615,Ninguna
953,FIN-8954,Risaralda,Geisha,Lavado Tradicional,1.5,11000.0,1812,Rainforest Alliance
984,FIN-8985,Chinchiná,Geisha,Lavado Tradicional,12.0,1200.0,1780,Fairtrade


## 10. Imputación de valores faltantes (`.fillna()`)

- `Area_Hectareas`: los `NaN` se sustituyen por su **promedio general**.
- `Kilos_Recolectados`: los `NaN` se completan con la **constante 0.0**.


In [167]:
print("NaN antes de imputar:")
df_fincas[["Area_Hectareas", "Kilos_Recolectados"]].isna().sum()


NaN antes de imputar:


Area_Hectareas        55
Kilos_Recolectados    39
dtype: int64

In [168]:
promedio_area = df_fincas["Area_Hectareas"].mean()
print(f"\nPromedio general de Area_Hectareas (ignora NaN): {promedio_area:,.4f} ha")

df_fincas["Area_Hectareas"] = df_fincas["Area_Hectareas"].fillna(promedio_area)
df_fincas["Kilos_Recolectados"] = df_fincas["Kilos_Recolectados"].fillna(0.0)

print("\nNaN después de imputar:")
df_fincas[["Area_Hectareas", "Kilos_Recolectados"]].isna().sum()



Promedio general de Area_Hectareas (ignora NaN): 5.8148 ha

NaN después de imputar:


Area_Hectareas        0
Kilos_Recolectados    0
dtype: int64

## 11. Transformación y creación de columnas con `.apply()`

Función de Python que calcula el **rendimiento por hectárea** (`Kilos_Recolectados / Area_Hectareas`) y se aplica fila a fila.


In [169]:
def calcular_rendimiento(fila):
    """Rendimiento en kilos por hectárea: Kilos_Recolectados / Area_Hectareas."""
    area = fila["Area_Hectareas"]
    if pd.isna(area) or area == 0:
        return float("nan")
    return fila["Kilos_Recolectados"] / area


df_fincas["Rendimiento_Kilos_Ha"] = df_fincas.apply(calcular_rendimiento, axis=1)

print("Columna creada: Rendimiento_Kilos_Ha")
df_fincas[["ID_Finca", "Area_Hectareas", "Kilos_Recolectados", "Rendimiento_Kilos_Ha"]].head(10)


Columna creada: Rendimiento_Kilos_Ha


,ID_Finca,Area_Hectareas,Kilos_Recolectados,Rendimiento_Kilos_Ha
0,FIN-8001,3.0,4800.0,1600.000000
1,FIN-8002,5.5,2500.0,454.545455
2,FIN-8003,1.5,1200.0,800.000000
3,FIN-8004,3.0,4800.0,1600.000000
4,FIN-8005,8.0,16000.0,2000.000000
5,FIN-8006,12.0,7500.0,625.000000
6,FIN-8007,5.5,4800.0,872.727273
7,FIN-8008,5.5,7500.0,1363.636364
8,FIN-8009,1.5,4800.0,3200.000000
9,FIN-8010,5.5,7500.0,1363.636364


In [170]:
print("Estadísticas del rendimiento (kg/ha):")
df_fincas["Rendimiento_Kilos_Ha"].describe()


Estadísticas del rendimiento (kg/ha):


count     1000.000000
mean      1422.052867
std       1739.089747
min          0.000000
25%        400.000000
50%        833.333333
75%       1666.666667
max      10666.666667
Name: Rendimiento_Kilos_Ha, dtype: float64

## 12. Renombrado y reordenamiento de atributos

- `.rename()`: `Metodo_Beneficio` → `Beneficio` y `Altitud_MSNM` → `Altitud`.
- Reordenamiento: `ID_Finca` y `Rendimiento_Kilos_Ha` al principio.


In [171]:
df_fincas = df_fincas.rename(
    columns={"Metodo_Beneficio": "Beneficio", "Altitud_MSNM": "Altitud"}
)

primeras = ["ID_Finca", "Rendimiento_Kilos_Ha"]
resto = [c for c in df_fincas.columns if c not in primeras]
df_fincas = df_fincas[primeras + resto]

print("Columnas resultantes (en orden):")
for i, c in enumerate(df_fincas.columns, start=1):
    print(f"  {i:>2}. {c}")

df_fincas.head()


Columnas resultantes (en orden):
   1. ID_Finca
   2. Rendimiento_Kilos_Ha
   3. Municipio
   4. Variedad
   5. Beneficio
   6. Area_Hectareas
   7. Kilos_Recolectados
   8. Altitud
   9. Certificacion


,ID_Finca,Rendimiento_Kilos_Ha,Municipio,Variedad,Beneficio,Area_Hectareas,Kilos_Recolectados,Altitud,Certificacion
0,FIN-8001,1600.000000,Chinchiná,Cenicafé 1,Honey,3.0,4800.0,1500,Ninguna
1,FIN-8002,454.545455,Manizales,Cenicafé 1,Natural,5.5,2500.0,1284,Rainforest Alliance
2,FIN-8003,800.000000,Manizales,Cenicafé 1,Lavado Tradicional,1.5,1200.0,1959,Rainforest Alliance
3,FIN-8004,1600.000000,Risaralda,Cenicafé 1,Lavado Tradicional,3.0,4800.0,1767,Ninguna
4,FIN-8005,2000.000000,Chinchiná,Castillo,Natural,8.0,16000.0,1929,Fairtrade


## 13. Agrupación y métricas de producción por Municipio (`.agg()`)


In [173]:
df_fincas.groupby("Municipio").agg(
    Altitud_Promedio=("Altitud", "mean"),
    Kilos_Totales=("Kilos_Recolectados", "sum")
)


,Altitud_Promedio,Kilos_Totales
Municipio,,
Anserma,1637.025773,882900.0
Chinchiná,1625.582011,916700.0
Manizales,1637.788177,959600.0
Palestina,1638.493213,1111500.0
Risaralda,1647.020725,978700.0


In [ ]:
top_municipio = metricas_municipio.loc[metricas_municipio["Kilos_Total"].idxmax()]
print("Municipio más productivo:", top_municipio["Municipio"])
print(f"Producción total: {top_municipio['Kilos_Total']:,.0f} kg")


Municipio más productivo: Palestina
Producción total: 1,111,500 kg


## 14. Dispersión con función personalizada (`.agg()`)

Función `rango` (`series.max() - series.min()`) aplicada a `Area_Hectareas` y `Kilos_Recolectados` tras agrupar por `Variedad`.


In [174]:
def rango(series):
    return series.max() - series.min()

df_fincas.groupby("Variedad")[["Area_Hectareas", "Kilos_Recolectados"]].agg(rango)


,Area_Hectareas,Kilos_Recolectados
Variedad,,
Borbon Rosado,17.0,16000.0
Castillo,17.0,16000.0
Cenicafé 1,17.0,16000.0
Colombia,17.0,16000.0
Geisha,17.0,16000.0


## 15. Agrupamiento multinivel (Municipio + Variedad)


In [175]:
df_fincas.groupby(["Municipio", "Variedad"]).agg(
    Altitud_Promedio=("Altitud", "mean"),
    Kilos_Totales=("Kilos_Recolectados", "sum")
)


Altitud_Promedio  Kilos_Totales
Municipio Variedad                                      
Anserma   Borbon Rosado       1664.538462       177100.0
          Castillo            1587.000000       215300.0
          Cenicafé 1          1599.600000        85300.0
          Colombia            1645.613636       191000.0
          Geisha              1677.166667       214200.0
Chinchiná Borbon Rosado       1620.673469       255100.0
          Castillo            1651.000000       237200.0
          Cenicafé 1          1567.882353       172400.0
          Colombia            1691.500000       115000.0
          Geisha              1606.857143       137000.0
Manizales Borbon Rosado       1703.341463       193800.0
          Castillo            1653.488889       215700.0
          Cenicafé 1          1616.000000       170700.0
          Colombia            1627.538462       200100.0
          Geisha              1581.000000       179300.0
Palestina Borbon Rosado       1629.276596       223800.0
          Castillo            1627.820000       265300.0
          Cenicafé 1          1650.300000       248300.0
          Colombia            1645.136364       215200.0
          Geisha              1641.300000       158900.0
Risaralda Borbon Rosado       1707.909091       138500.0
          Castillo            1686.868421       171900.0
          Cenicafé 1          1626.976744       239100.0
          Colombia            1612.736842       213800.0
          Geisha              1613.878049       215400.0

In [176]:
print("Las 10 combinaciones con mayor producción:")
metricas_multinivel.sort_values("Kilos_Total", ascending=False).head(10)


Las 10 combinaciones con mayor producción:


,Municipio,Variedad,Altitud_Promedio,Kilos_Total
16,Palestina,Castillo,1627.820000,265300.0
5,Chinchiná,Borbon Rosado,1620.673469,255100.0
17,Palestina,Cenicafé 1,1650.300000,248300.0
22,Risaralda,Cenicafé 1,1626.976744,239100.0
6,Chinchiná,Castillo,1651.000000,237200.0
15,Palestina,Borbon Rosado,1629.276596,223800.0
11,Manizales,Castillo,1653.488889,215700.0
24,Risaralda,Geisha,1613.878049,215400.0
1,Anserma,Castillo,1587.000000,215300.0
18,Palestina,Colombia,1645.136364,215200.0


## Resumen del DataFrame final


In [177]:
df_fincas.info()
print("\nValores faltantes restantes:", int(df_fincas.isna().sum().sum()))


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID_Finca              1000 non-null   str    
 1   Rendimiento_Kilos_Ha  1000 non-null   float64
 2   Municipio             1000 non-null   str    
 3   Variedad              1000 non-null   str    
 4   Beneficio             1000 non-null   str    
 5   Area_Hectareas        1000 non-null   float64
 6   Kilos_Recolectados    1000 non-null   float64
 7   Altitud               1000 non-null   int64  
 8   Certificacion         1000 non-null   str    
dtypes: float64(3), int64(1), str(5)
memory usage: 70.4 KB

Valores faltantes restantes: 0
